# Travel project — run local UI

Run the cells below in order. Streamlit starts at **http://localhost:8501** and opens as an **IDE Simple Browser tab** (not Chrome).

If no tab appears: **Terminal → Run Task… → Open Travel Atlas UI**

Sidebar: **Home** → **Pin Intelligence** → **Atlas Explorer**

In [ ]:
from pathlib import Path
import subprocess
import sys

TRAVEL_ROOT = Path.cwd()
if not (TRAVEL_ROOT / "app_home.py").exists():
    # Allow opening the notebook from travel_code/ instead of travel/
    candidate = TRAVEL_ROOT / "travel"
    if (candidate / "app_home.py").exists():
        TRAVEL_ROOT = candidate

TRAVEL_CODE_ROOT = TRAVEL_ROOT.parent
APP = TRAVEL_ROOT / "app_home.py"
REQUIREMENTS = TRAVEL_ROOT / "requirements.txt"
BOOTSTRAP = TRAVEL_ROOT / "bootstrap_demo.py"

assert APP.exists(), f"Missing {APP} — open this notebook from the travel/ folder"

sys.path.insert(0, str(TRAVEL_ROOT))
from travel_atlas.env_loader import load_travel_env

_env_path = load_travel_env(env_path=TRAVEL_ROOT / ".env")
print("travel root:", TRAVEL_ROOT)
print("app:", APP)
print("env loaded:", "yes" if _env_path else "no (.env missing)")

## 1. Install dependencies

In [ ]:
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REQUIREMENTS)],
)
print("Dependencies installed.")

## 2. Bootstrap demo pins (skipped if `travel_pins.db` already has data)

In [ ]:
subprocess.check_call([sys.executable, str(BOOTSTRAP)], cwd=str(TRAVEL_ROOT))
print("Bootstrap done.")

## 3. Start Streamlit

Starts headless (no Chrome). Opens Cursor **Simple Browser** as an editor tab, and embeds a preview below.

In [ ]:
import os
import shutil
import time
import urllib.error
import urllib.request
from urllib.parse import quote

UI_URL = "http://localhost:8501"
UI_PORT = "8501"
TASK_LABEL = "Open Travel Atlas UI"


def _open_in_ide_browser(url: str) -> str:
    """Open Simple Browser inside Cursor. Never use os.startfile (opens Chrome)."""
    encoded = quote(url, safe="")
    simple = f"vscode://vscode.simple-browser/show?url={encoded}"
    cursor = shutil.which("cursor")
    if not cursor:
        return (
            f"`cursor` CLI not on PATH. Run task `{TASK_LABEL}` or: "
            f"Ctrl+Shift+P → Simple Browser: Show → {url}"
        )
    # Hand the vscode:// URI to Cursor itself (IDE handler → Simple Browser tab).
    # Do NOT use os.startfile / Windows shell — that routes the URL to Chrome.
    try:
        subprocess.Popen(
            [cursor, "--reuse-window", simple],
            cwd=str(TRAVEL_ROOT),
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        return "Asked Cursor to open Simple Browser tab"
    except OSError as exc:
        return (
            f"Could not launch Cursor CLI ({exc}). "
            f"Terminal → Run Task… → `{TASK_LABEL}`"
        )


def _wait_until_up(url: str, proc: subprocess.Popen, timeout_s: float = 30.0) -> None:
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if proc.poll() is not None:
            raise RuntimeError(f"Streamlit exited early with code {proc.returncode}")
        try:
            with urllib.request.urlopen(url, timeout=1) as resp:
                if getattr(resp, "status", 200) < 500:
                    return
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            time.sleep(0.4)
    raise TimeoutError(f"Streamlit did not become ready at {url}")


prev = globals().get("_streamlit_proc")
if prev is not None and prev.poll() is None:
    prev.terminate()
    try:
        prev.wait(timeout=5)
    except subprocess.TimeoutExpired:
        prev.kill()
    print("Stopped previous Streamlit process.")

env = os.environ.copy()
env.setdefault("STREAMLIT_BROWSER_GATHER_USAGE_STATS", "false")
env["BROWSER"] = "none"
env["STREAMLIT_SERVER_HEADLESS"] = "true"
# Ensure imports resolve whether cwd is travel/ or travel_code/
env["PYTHONPATH"] = os.pathsep.join(
    [str(TRAVEL_ROOT), str(TRAVEL_CODE_ROOT), env.get("PYTHONPATH", "")]
)

_streamlit_proc = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        str(APP),
        "--server.headless",
        "true",
        "--browser.gatherUsageStats",
        "false",
        "--server.port",
        UI_PORT,
    ],
    cwd=str(TRAVEL_ROOT),
    env=env,
)

_wait_until_up(UI_URL, _streamlit_proc)
how = _open_in_ide_browser(UI_URL)

print(f"UI ready at {UI_URL} (Streamlit headless — no Chrome from Streamlit)")
print(how)
print(f"pid={_streamlit_proc.pid}")
print(f"Fallback: Terminal → Run Task… → {TASK_LABEL}")

try:
    from IPython.display import HTML, display

    # Preview stays in the notebook (IDE). Avoid clickable http:// markdown — opens Chrome.
    display(
        HTML(
            f'<p style="margin:0 0 8px 0;opacity:.8">In-notebook preview</p>'
            f'<iframe src="{UI_URL}" width="100%" height="720" '
            f'style="border:1px solid #444;border-radius:6px;" '
            f'title="Travel Atlas"></iframe>'
        )
    )
except Exception:
    pass

## Stop server (optional)

prev = globals().get("_streamlit_proc")
if prev is None or prev.poll() is not None:
    print("No running Streamlit process from this notebook.")
else:
    prev.terminate()
    try:
        prev.wait(timeout=5)
    except subprocess.TimeoutExpired:
        prev.kill()
    print("Streamlit stopped.")